# Death Clock — Longevity Prediction Engine

A full pipeline for predicting life expectancy from a 29-question lifestyle survey, anchored to SSA actuarial life tables and optionally calibrated on NHANES data.

**Sections:**
1. Setup & Config
2. Questionnaire Schema
3. Normalization Helpers
4. Feature Engineering
5. SSA Baseline Life Tables
6. Vitality Model
7. Cox Proportional Hazards (NHANES)
8. Paper Ingestion Pipeline
9. Chunking, Embeddings & Vector DB
10. Retrieval & Explanation Assembly
11. Evaluation & Export

## Section 1: Setup & Config

Creates folder structure and sets global configuration. Directories:
- `raw/` — downloaded datasets (NHANES, life tables, papers)
- `processed/` — cleaned intermediate files
- `output/` — final results, model coefficients, exports

In [ ]:
# !pip install lifelines sentence-transformers chromadb pymed requests pandas numpy

In [ ]:
import json
import re
import math
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/content")
RAW  = ROOT / "raw"
PROC = ROOT / "processed"
OUT  = ROOT / "output"

for p in [RAW, PROC, OUT]:
    p.mkdir(parents=True, exist_ok=True)

# Put your own values here.
# NCBI API key is free and raises rate limit from 3 to 10 req/s.
# Get one at: https://www.ncbi.nlm.nih.gov/account/settings/
CONFIG = {
    "email": "your_email@example.com",
    "ncbi_api_key": "",
    "random_seed": 42,
}

np.random.seed(CONFIG["random_seed"])
print("✓ Section 1 complete — workspace ready")

## Section 2: Questionnaire Schema

Defines the "contract" between the iOS frontend and the Python backend. Every field has a `type`, allowed `values`, and a `required` flag. Mirrors Death Clock's 29-step survey exactly.

In [ ]:
QUESTIONNAIRE = {
    # ── Demographics ──
    "first_name":    {"type": "str",   "required": True, "max_length": 60},
    "age":           {"type": "int",   "required": True, "min": 18, "max": 120},
    "sex":           {"type": "enum",  "required": True, "values": ["male", "female"]},
    "race": {
        "type": "enum",
        "required": True,
        "values": ["asian", "black", "hispanic", "american_indian_alaska_native", "white"],
    },
    "in_us":         {"type": "bool",  "required": True},
    "height_value":  {"type": "float", "required": True, "min": 0.5, "max": 250.0},
    "height_unit":   {"type": "enum",  "required": True, "values": ["cm", "ft_in", "m"]},
    "weight_value":  {"type": "float", "required": True, "min": 20.0, "max": 500.0},
    "weight_unit":   {"type": "enum",  "required": True, "values": ["kg", "lb"]},

    # ── Diet (4 questions) ──
    "diet_fruits_veggies": {
        "type": "enum", "required": True,
        "values": ["rarely_or_never", "several_times_a_week", "daily", "five_or_more_servings_a_day"],
    },
    "diet_processed_foods": {
        "type": "enum", "required": True,
        "values": ["daily", "several_times_a_week_or_more", "once_a_week", "rarely_or_never"],
    },
    "diet_sugar": {
        "type": "enum", "required": True,
        "values": ["sweets_several_times_a_day", "daily_sweet_treat", "just_a_few_treats_a_week", "none"],
    },
    "diet_water": {
        "type": "enum", "required": True,
        "values": ["less_than_one_glass", "two_to_five_glasses", "six_to_nine_glasses", "ten_or_more_glasses"],
    },

    # ── Exercise (4 questions) ──
    "exercise_cardio": {
        "type": "enum", "required": True,
        "values": ["rarely_or_never", "less_than_150_minutes", "150_to_300_minutes", "more_than_300_minutes"],
    },
    "exercise_weights": {
        "type": "enum", "required": True,
        "values": ["rarely_or_never", "less_than_once_a_week", "one_to_two_days_per_week", "more_than_two_days_per_week"],
    },
    "exercise_mobility": {
        "type": "enum", "required": True,
        "values": ["never", "a_few_times_a_month", "one_to_two_times_per_week", "three_or_more_times_per_week"],
    },
    "exercise_sitting": {
        "type": "enum", "required": True,
        "values": ["more_than_8_hours", "four_to_eight_hours", "two_to_four_hours", "less_than_2_hours"],
    },

    # ── Activity Tracking (1 question) ──
    "activity_tracking": {
        "type": "enum", "required": True,
        "values": ["yes_tracking_both", "only_tracking_activity", "only_tracking_sleep", "no_not_using_any_device"],
    },

    # ── Sleep (2 questions) ──
    "sleep_duration": {
        "type": "enum", "required": True,
        "values": ["never", "one_to_two_nights_per_week", "three_to_four_nights_per_week", "five_or_more_nights_per_week"],
    },
    "sleep_trouble": {
        "type": "enum", "required": True,
        "values": ["five_or_more_nights_per_week", "three_to_four_nights_per_week", "one_to_two_nights_per_week", "never"],
    },

    # ── Community (4 questions) ──
    "community_time": {
        "type": "enum", "required": True,
        "values": ["never", "a_few_times_a_month", "weekly", "daily"],
    },
    "relationship_status": {
        "type": "enum", "required": True,
        "values": ["in_a_long_term_relationship", "married", "single", "divorced_widowed_separated"],
    },
    "children": {
        "type": "enum", "required": True,
        "values": ["yes_not_living_at_home", "yes_living_at_home", "no_but_planning_to", "no_and_not_planning_to"],
    },
    "social_support": {
        "type": "enum", "required": True,
        "values": ["not_supportive", "slightly_supportive", "fairly_supportive", "completely_supportive"],
    },

    # ── Health Planning (3 questions) ──
    "household_income": {
        "type": "enum", "required": True,
        "values": ["under_75k", "75k_200k", "200k_500k", "over_500k"],
    },
    "bloodwork_recency": {
        "type": "enum", "required": True,
        "values": [
            "eager_to_get_blood_work_done_soon",
            "got_blood_work_recently_will_keep_doing_it",
            "rarely_get_blood_work_done_dont_see_a_need",
            "avoid_blood_work",
        ],
    },
    "clinical_data_method": {
        "type": "enum", "required": True,
        "values": ["upload_blood_work_or_lab_results", "sync_my_medical_records", "both_upload_and_sync", "none_of_the_above"],
    },

    # ── Substance Use (2 questions) ──
    "alcohol": {
        "type": "enum", "required": True,
        "values": ["15_or_more_drinks_per_week", "8_14_drinks_per_week", "1_7_drinks_per_week", "dont_drink"],
    },
    "nicotine": {
        "type": "enum", "required": True,
        "values": ["current_daily_user", "current_occasional_user", "former_user", "never_used"],
    },

    # ── Mental Health (2 questions) ──
    "stress": {
        "type": "enum", "required": True,
        "values": ["almost_all_the_time", "frequently", "occasionally", "rarely_or_never"],
    },
    "mental_health_impact": {
        "type": "enum", "required": True,
        "values": ["severely", "moderately", "mildly", "not_at_all"],
    },

    # ── Healthcare (2 questions) ──
    "checkups": {
        "type": "enum", "required": True,
        "values": ["never", "every_five_years", "every_two_to_three_years", "yearly"],
    },
    "cancer_screenings": {
        "type": "enum", "required": True,
        "values": ["never", "once_or_twice", "sometimes_but_not_consistently", "as_recommended"],
    },

    # ── Family History (1 question) ──
    "grandparents_max_age": {
        "type": "enum", "required": True,
        "values": ["under_70", "70_79", "80_89", "90_plus"],
    },

    # ── Disease / Clinical (5 questions) ──
    "overweight": {
        "type": "enum", "required": True,
        "values": ["obese", "overweight", "a_little_overweight", "no"],
    },
    "blood_pressure": {
        "type": "enum", "required": True,
        "values": ["below_120_80_normal", "i_dont_know", "120_80_to_139_89_elevated", "140_90_or_higher_high"],
    },
    "ldl": {
        "type": "enum", "required": True,
        "values": ["below_100_optimal", "i_dont_know", "100_159_borderline_high", "160_or_higher_high"],
    },
    "glucose": {
        "type": "enum", "required": True,
        "values": ["below_100_normal", "i_dont_know", "100_125_prediabetes", "126_or_higher_diabetes"],
    },
    "chronic_disease": {
        "type": "enum", "required": True,
        "values": ["yes", "risk_factors_for_chronic_diseases", "im_not_sure", "no"],
    },
}

with open(PROC / "questionnaire_schema.json", "w") as f:
    json.dump(QUESTIONNAIRE, f, indent=2)

print(f"✓ Section 2 complete — {len(QUESTIONNAIRE)} fields defined")

## Section 3: Normalization Helpers

Converts messy user inputs into clean canonical form before any math:
1. Heights → meters
2. Weights → kilograms
3. BMI computation
4. Text labels → snake_case tokens
5. Required-field validation

In [ ]:
def normalize_answer_label(text):
    """
    Convert any human-readable answer into a canonical snake_case token.

    Examples:
        "Five or more servings a day" → "five_or_more_servings_a_day"
        "I don't know"               → "i_dont_know"
        "120/80 to 139/89 (elevated)" → "120_80_to_139_89_elevated"
    """
    if text is None:
        return None
    t = str(text).strip().lower()
    t = t.replace("&", "and")
    t = t.replace("'", "")          # don't → dont
    t = t.replace("/", " ")         # 120/80 → 120 80
    t = re.sub(r"[^a-z0-9\s]+", "", t)
    t = re.sub(r"\s+", "_", t).strip("_")
    return t


def normalize_height(value, unit):
    """
    Convert any height input to meters.

    Args:
        value: number (for cm/m) or [feet, inches] tuple (for ft_in)
        unit: "cm", "m", or "ft_in"

    Examples:
        normalize_height(175, "cm")        → 1.75
        normalize_height(1.75, "m")        → 1.75
        normalize_height([5, 11], "ft_in") → 1.8034
    """
    unit = normalize_answer_label(unit)
    if unit == "cm":
        return float(value) / 100.0
    if unit == "m":
        return float(value)
    if unit == "ft_in":
        if isinstance(value, (tuple, list)) and len(value) == 2:
            ft, inch = value
            return (float(ft) * 12.0 + float(inch)) * 0.0254
        raise ValueError("height_value must be [feet, inches] when unit='ft_in'")
    raise ValueError(f"Unknown height unit: {unit}")


def normalize_weight(value, unit):
    """
    Convert any weight input to kilograms.

    Examples:
        normalize_weight(180, "lb") → 81.65
        normalize_weight(80, "kg")  → 80.0
    """
    unit = normalize_answer_label(unit)
    if unit == "kg":
        return float(value)
    if unit == "lb":
        return float(value) * 0.45359237
    raise ValueError(f"Unknown weight unit: {unit}")


def compute_bmi(height_m, weight_kg):
    """
    BMI = weight(kg) / height(m)²

    Categories:
        < 18.5  → Underweight
        18.5-24.9 → Normal
        25-29.9   → Overweight
        30-34.9   → Obese I
        35-39.9   → Obese II
        40+       → Obese III
    """
    if height_m <= 0:
        raise ValueError("Height must be positive")
    return round(weight_kg / (height_m ** 2), 1)


def validate_required_fields(answers):
    """Check that all required questionnaire fields are present and non-empty."""
    required = [k for k, v in QUESTIONNAIRE.items() if v.get("required", True)]
    missing = [k for k in required if k not in answers or answers[k] in [None, "", []]]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    return True


def clamp(x, lo, hi):
    """Keep x within [lo, hi] range."""
    return max(lo, min(hi, x))


# Quick sanity tests
assert normalize_answer_label("Five or more servings a day") == "five_or_more_servings_a_day"
assert normalize_answer_label("I don't know") == "i_dont_know"
assert abs(normalize_height(175, "cm") - 1.75) < 0.001
assert abs(normalize_height([5, 11], "ft_in") - 1.8034) < 0.01
assert abs(normalize_weight(180, "lb") - 81.65) < 0.1
assert compute_bmi(1.75, 80) == 26.1

print("✓ Section 3 complete — all normalization helpers pass tests")

## Section 4: Questionnaire → Feature Engineering

Converts survey text answers into numeric features on a **0–3 scale**:
- For *good* factors (exercise, sleep quality): `3 = best`
- For *bad* factors (smoking, stress, processed food): `3 = worst / highest risk`

The vitality model in Section 6 handles the sign convention correctly.

In [ ]:
def score_choice(raw_answer, score_map):
    """
    Convert a raw answer string to its numeric score.

    Args:
        raw_answer: the user's answer (any casing/format)
        score_map: dict mapping canonical tokens → scores
    """
    key = normalize_answer_label(raw_answer)
    if key not in score_map:
        for k in score_map:
            if k in key or key in k:
                return score_map[k]
        raise ValueError(
            f"Invalid answer '{raw_answer}' (normalized: '{key}'). "
            f"Valid: {list(score_map.keys())}"
        )
    return score_map[key]


SCORE_MAPS = {
    # DIET — good
    "diet_fruits_veggies": {
        "rarely_or_never": 0, "several_times_a_week": 1,
        "daily": 2, "five_or_more_servings_a_day": 3,
    },
    # DIET — bad (higher = more risk)
    "diet_processed_foods": {
        "daily": 3, "several_times_a_week_or_more": 2,
        "once_a_week": 1, "rarely_or_never": 0,
    },
    "diet_sugar": {
        "sweets_several_times_a_day": 3, "daily_sweet_treat": 2,
        "just_a_few_treats_a_week": 1, "none": 0,
    },
    "diet_water": {
        "less_than_one_glass": 0, "two_to_five_glasses": 1,
        "six_to_nine_glasses": 2, "ten_or_more_glasses": 3,
    },

    # EXERCISE — good
    "exercise_cardio": {
        "rarely_or_never": 0, "less_than_150_minutes": 1,
        "150_to_300_minutes": 2, "more_than_300_minutes": 3,
    },
    "exercise_weights": {
        "rarely_or_never": 0, "less_than_once_a_week": 1,
        "one_to_two_days_per_week": 2, "more_than_two_days_per_week": 3,
    },
    "exercise_mobility": {
        "never": 0, "a_few_times_a_month": 1,
        "one_to_two_times_per_week": 2, "three_or_more_times_per_week": 3,
    },
    # SITTING — bad
    "exercise_sitting": {
        "more_than_8_hours": 3, "four_to_eight_hours": 2,
        "two_to_four_hours": 1, "less_than_2_hours": 0,
    },

    "activity_tracking": {
        "yes_tracking_both": 3, "only_tracking_activity": 2,
        "only_tracking_sleep": 1, "no_not_using_any_device": 0,
    },

    # SLEEP — good
    "sleep_duration": {
        "never": 0, "one_to_two_nights_per_week": 1,
        "three_to_four_nights_per_week": 2, "five_or_more_nights_per_week": 3,
    },
    # SLEEP TROUBLE — bad
    "sleep_trouble": {
        "five_or_more_nights_per_week": 3, "three_to_four_nights_per_week": 2,
        "one_to_two_nights_per_week": 1, "never": 0,
    },

    # COMMUNITY — good
    "community_time": {
        "never": 0, "a_few_times_a_month": 1, "weekly": 2, "daily": 3,
    },
    "relationship_status": {
        "single": 0, "divorced_widowed_separated": 1,
        "in_a_long_term_relationship": 3, "married": 3,
    },
    "children": {
        "no_and_not_planning_to": 0, "no_but_planning_to": 1,
        "yes_living_at_home": 1, "yes_not_living_at_home": 2,
    },
    "social_support": {
        "not_supportive": 0, "slightly_supportive": 1,
        "fairly_supportive": 2, "completely_supportive": 3,
    },

    # HEALTH PLANNING
    "household_income": {
        "under_75k": 0, "75k_200k": 1, "200k_500k": 2, "over_500k": 3,
    },
    "bloodwork_recency": {
        "avoid_blood_work": 0, "eager_to_get_blood_work_done_soon": 0,
        "rarely_get_blood_work_done_dont_see_a_need": 1,
        "got_blood_work_recently_will_keep_doing_it": 3,
    },
    "clinical_data_method": {
        "none_of_the_above": 0, "upload_blood_work_or_lab_results": 1,
        "sync_my_medical_records": 1, "both_upload_and_sync": 2,
    },

    # SUBSTANCE USE — bad
    "alcohol": {
        "dont_drink": 0, "1_7_drinks_per_week": 1,
        "8_14_drinks_per_week": 2, "15_or_more_drinks_per_week": 3,
    },
    "nicotine": {
        "never_used": 0, "former_user": 1,
        "current_occasional_user": 2, "current_daily_user": 3,
    },

    # MENTAL HEALTH — bad
    "stress": {
        "rarely_or_never": 0, "occasionally": 1,
        "frequently": 2, "almost_all_the_time": 3,
    },
    "mental_health_impact": {
        "not_at_all": 0, "mildly": 1, "moderately": 2, "severely": 3,
    },

    # HEALTHCARE — good
    "checkups": {
        "never": 0, "every_five_years": 1,
        "every_two_to_three_years": 2, "yearly": 3,
    },
    "cancer_screenings": {
        "never": 0, "once_or_twice": 1,
        "sometimes_but_not_consistently": 2, "as_recommended": 3,
    },

    # FAMILY HISTORY — good
    "grandparents_max_age": {
        "under_70": 0, "70_79": 1, "80_89": 2, "90_plus": 3,
    },

    # DISEASE — bad
    "overweight": {
        "no": 0, "a_little_overweight": 1, "overweight": 2, "obese": 3,
    },
    "blood_pressure": {
        "below_120_80_normal": 0, "i_dont_know": 1,
        "120_80_to_139_89_elevated": 2, "140_90_or_higher_high": 3,
    },
    "ldl": {
        "below_100_optimal": 0, "i_dont_know": 1,
        "100_159_borderline_high": 2, "160_or_higher_high": 3,
    },
    "glucose": {
        "below_100_normal": 0, "i_dont_know": 1,
        "100_125_prediabetes": 2, "126_or_higher_diabetes": 3,
    },
    "chronic_disease": {
        "no": 0, "im_not_sure": 1,
        "risk_factors_for_chronic_diseases": 2, "yes": 3,
    },
}

In [ ]:
def normalize_questionnaire(answers):
    """Clean raw answers → canonical form + compute BMI. Runs BEFORE scoring."""
    validate_required_fields(answers)

    out = dict(answers)
    out["first_name"] = str(answers["first_name"]).strip()
    out["age"]        = int(answers["age"])
    out["sex"]        = normalize_answer_label(answers["sex"])
    out["race"]       = normalize_answer_label(answers["race"])
    out["in_us"]      = bool(answers["in_us"])
    out["height_m"]   = normalize_height(answers["height_value"], answers["height_unit"])
    out["weight_kg"]  = normalize_weight(answers["weight_value"], answers["weight_unit"])
    out["bmi"]        = compute_bmi(out["height_m"], out["weight_kg"])
    return out


def questionnaire_to_features(answers):
    """
    Convert normalized answers → full numeric feature vector.

    Returns a dict with:
      - Demographics (one-hot encoded)
      - Physical measurements (height_m, weight_kg, bmi)
      - 31 lifestyle/health scores (0-3 scale each)
    """
    a = normalize_questionnaire(answers)

    features = {
        "first_name":    a["first_name"],
        "age":           a["age"],
        "sex_male":      1 if a["sex"] == "male" else 0,
        "sex_female":    1 if a["sex"] == "female" else 0,
        "race_asian":    1 if a["race"] == "asian" else 0,
        "race_black":    1 if a["race"] == "black" else 0,
        "race_hispanic": 1 if a["race"] == "hispanic" else 0,
        "race_ai_an":    1 if a["race"] == "american_indian_alaska_native" else 0,
        "race_white":    1 if a["race"] == "white" else 0,
        "in_us":         1 if a["in_us"] else 0,
        "height_m":      a["height_m"],
        "weight_kg":     a["weight_kg"],
        "bmi":           a["bmi"],
    }

    score_field_map = {
        "diet_fruits_veggies":  "fruitveg_score",
        "diet_processed_foods": "processed_food_score",
        "diet_sugar":           "sugar_score",
        "diet_water":           "water_score",
        "exercise_cardio":      "cardio_score",
        "exercise_weights":     "weights_score",
        "exercise_mobility":    "stretch_score",
        "exercise_sitting":     "sitting_score",
        "activity_tracking":    "activity_tracking_score",
        "sleep_duration":       "sleep_duration_score",
        "sleep_trouble":        "sleep_disturbance_score",
        "community_time":       "community_time_score",
        "relationship_status":  "relationship_score",
        "children":             "children_score",
        "social_support":       "support_score",
        "household_income":     "income_score",
        "bloodwork_recency":    "bloodwork_score",
        "clinical_data_method": "clinical_data_method_score",
        "alcohol":              "alcohol_score",
        "nicotine":             "nicotine_score",
        "stress":               "stress_score",
        "mental_health_impact": "mental_health_score",
        "checkups":             "checkup_score",
        "cancer_screenings":    "screening_score",
        "grandparents_max_age": "grandparent_longevity_score",
        "overweight":           "overweight_score",
        "blood_pressure":       "bp_score",
        "ldl":                  "ldl_score",
        "glucose":              "glucose_score",
        "chronic_disease":      "chronic_disease_score",
    }

    for q_field, feat_name in score_field_map.items():
        features[feat_name] = score_choice(a[q_field], SCORE_MAPS[q_field])

    return features


FEATURE_NAMES = [
    "age", "sex_male", "sex_female",
    "race_asian", "race_black", "race_hispanic", "race_ai_an", "race_white",
    "in_us", "height_m", "weight_kg", "bmi",
    "fruitveg_score", "processed_food_score", "sugar_score", "water_score",
    "cardio_score", "weights_score", "stretch_score", "sitting_score",
    "activity_tracking_score", "sleep_duration_score", "sleep_disturbance_score",
    "community_time_score", "relationship_score", "children_score",
    "income_score", "support_score", "alcohol_score", "nicotine_score",
    "stress_score", "mental_health_score", "checkup_score", "bloodwork_score",
    "clinical_data_method_score", "screening_score", "grandparent_longevity_score",
    "overweight_score", "bp_score", "ldl_score", "glucose_score",
    "chronic_disease_score",
]

with open(PROC / "features_definition.json", "w") as f:
    json.dump({"features": FEATURE_NAMES, "score_maps": SCORE_MAPS}, f, indent=2)

In [ ]:
SAMPLE_USER = {
    "first_name": "Amani",
    "age": 20,
    "sex": "male",
    "race": "black",
    "in_us": True,
    "height_value": [5, 10],
    "height_unit": "ft_in",
    "weight_value": 165,
    "weight_unit": "lb",
    "diet_fruits_veggies": "daily",
    "diet_processed_foods": "once_a_week",
    "diet_sugar": "just_a_few_treats_a_week",
    "diet_water": "six_to_nine_glasses",
    "exercise_cardio": "150_to_300_minutes",
    "exercise_weights": "one_to_two_days_per_week",
    "exercise_mobility": "a_few_times_a_month",
    "exercise_sitting": "four_to_eight_hours",
    "activity_tracking": "yes_tracking_both",
    "sleep_duration": "five_or_more_nights_per_week",
    "sleep_trouble": "one_to_two_nights_per_week",
    "community_time": "weekly",
    "relationship_status": "single",
    "children": "no_and_not_planning_to",
    "social_support": "fairly_supportive",
    "household_income": "under_75k",
    "bloodwork_recency": "eager_to_get_blood_work_done_soon",
    "clinical_data_method": "none_of_the_above",
    "alcohol": "1_7_drinks_per_week",
    "nicotine": "never_used",
    "stress": "occasionally",
    "mental_health_impact": "mildly",
    "checkups": "yearly",
    "cancer_screenings": "as_recommended",
    "grandparents_max_age": "80_89",
    "overweight": "no",
    "blood_pressure": "below_120_80_normal",
    "ldl": "i_dont_know",
    "glucose": "below_100_normal",
    "chronic_disease": "no",
}

sample_features = questionnaire_to_features(SAMPLE_USER)
print(f"\n✓ Section 4 complete — sample user '{sample_features['first_name']}':")
print(f"  BMI: {sample_features['bmi']}")
print(f"  Cardio score: {sample_features['cardio_score']}/3")
print(f"  Sleep score: {sample_features['sleep_duration_score']}/3")
print(f"  Nicotine score: {sample_features['nicotine_score']}/3 (0 = best)")

sample_df = pd.DataFrame([{k: v for k, v in sample_features.items() if k != "first_name"}])
sample_df.to_csv(OUT / "feature_matrix.csv", index=False)

## Section 5: SSA Baseline Life Tables

The SSA publishes `q(x)` — the probability that someone exactly age `x` will die within 1 year — for every age from 0 to 119, split by sex. This is the foundation that every other model section modifies.

**Source:** SSA 2022 Period Life Table (2025 Trustees Report)  
**URL:** https://www.ssa.gov/oact/STATS/table4c6.html

In [ ]:
# q(x) = probability of death within 1 year at exact age x
# 120 values: ages 0 through 119

SSA_QX_MALE = [
    0.006064,0.000491,0.000309,0.000248,0.000199,0.000167,0.000143,0.000126,
    0.000121,0.000121,0.000127,0.000143,0.000171,0.000227,0.000320,0.000451,
    0.000622,0.000826,0.001026,0.001182,0.001301,0.001404,0.001498,0.001586,
    0.001679,0.001776,0.001881,0.001985,0.002095,0.002219,0.002332,0.002445,
    0.002562,0.002653,0.002716,0.002791,0.002894,0.002994,0.003091,0.003217,
    0.003353,0.003499,0.003642,0.003811,0.003996,0.004175,0.004388,0.004666,
    0.004973,0.005305,0.005666,0.006069,0.006539,0.007073,0.007675,0.008348,
    0.009051,0.009822,0.010669,0.011548,0.012458,0.013403,0.014450,0.015571,
    0.016737,0.017897,0.019017,0.020213,0.021569,0.023088,0.024828,0.026705,
    0.028761,0.031116,0.033861,0.037088,0.041126,0.045241,0.049793,0.054768,
    0.060660,0.067027,0.073999,0.081737,0.090458,0.100525,0.111793,0.124494,
    0.138398,0.153207,0.169704,0.187963,0.208395,0.230808,0.253914,0.277402,
    0.300882,0.324326,0.347332,0.369430,0.391927,0.414726,0.437722,0.460800,
    0.483840,0.508032,0.533434,0.560105,0.588111,0.617516,0.648392,0.680812,
    0.714852,0.750595,0.788125,0.827531,0.868907,0.912353,0.957970,1.000000,
]

SSA_QX_FEMALE = [
    0.005119,0.000398,0.000240,0.000198,0.000160,0.000134,0.000118,0.000109,
    0.000106,0.000106,0.000111,0.000121,0.000140,0.000162,0.000188,0.000224,
    0.000276,0.000337,0.000395,0.000450,0.000496,0.000532,0.000567,0.000610,
    0.000650,0.000699,0.000743,0.000796,0.000855,0.000924,0.000988,0.001053,
    0.001123,0.001198,0.001263,0.001324,0.001403,0.001493,0.001596,0.001700,
    0.001803,0.001905,0.002009,0.002116,0.002223,0.002352,0.002516,0.002712,
    0.002936,0.003177,0.003407,0.003642,0.003917,0.004238,0.004619,0.005040,
    0.005493,0.005987,0.006509,0.007067,0.007658,0.008305,0.008991,0.009681,
    0.010343,0.011018,0.011743,0.012532,0.013512,0.014684,0.016025,0.017468,
    0.019195,0.021195,0.023452,0.025980,0.029153,0.032394,0.035888,0.039676,
    0.044156,0.049087,0.054635,0.061066,0.068431,0.076841,0.086205,0.096851,
    0.109019,0.121867,0.135805,0.151108,0.168020,0.186340,0.206432,0.228086,
    0.250406,0.273699,0.296984,0.319502,0.342716,0.366532,0.390844,0.415531,
    0.440463,0.466891,0.494904,0.524599,0.556075,0.589439,0.624805,0.662294,
    0.702031,0.744153,0.788125,0.827531,0.868907,0.912353,0.957970,1.000000,
]

SSA_EX_MALE = [
    74.74,74.20,73.23,72.25,71.27,70.29,69.30,68.31,67.32,66.32,65.33,64.34,
    63.35,62.36,61.37,60.39,59.42,58.46,57.50,56.56,55.63,54.70,53.78,52.86,
    51.94,51.03,50.12,49.21,48.31,47.41,46.51,45.62,44.73,43.84,42.96,42.08,
    41.19,40.31,39.43,38.55,37.67,36.80,35.93,35.05,34.19,33.32,32.46,31.60,
    30.75,29.90,29.05,28.22,27.39,26.56,25.75,24.94,24.15,23.37,22.59,21.83,
    21.08,20.34,19.61,18.89,18.18,17.48,16.79,16.11,15.43,14.76,14.09,13.44,
    12.80,12.16,11.53,10.92,10.32,9.74,9.18,8.64,8.11,7.60,7.11,6.64,6.18,
    5.75,5.34,4.94,4.58,4.23,3.91,3.60,3.32,3.06,2.83,2.63,2.44,2.28,2.13,2.00,
]

SSA_EX_FEMALE = [
    80.18,79.60,78.63,77.65,76.66,75.67,74.68,73.69,72.70,71.71,70.72,69.72,
    68.73,67.74,66.75,65.76,64.78,63.80,62.82,61.84,60.87,59.90,58.93,57.97,
    57.00,56.04,55.08,54.12,53.16,52.20,51.25,50.30,49.35,48.41,47.47,46.53,
    45.59,44.65,43.72,42.79,41.86,40.93,40.01,39.09,38.17,37.25,36.34,35.43,
    34.53,33.63,32.73,31.84,30.96,30.08,29.20,28.34,27.48,26.63,25.78,24.95,
    24.12,23.31,22.50,21.70,20.90,20.12,19.34,18.56,17.79,17.03,16.27,15.53,
    14.80,14.08,13.37,12.68,12.00,11.35,10.71,10.09,9.49,8.90,8.34,7.79,7.26,
    6.76,6.28,5.83,5.40,5.00,4.62,4.27,3.94,3.64,3.36,3.10,2.87,2.66,2.47,2.30,
]

In [ ]:
def get_baseline_qx(age, sex):
    """Get the baseline q(x) table from current age onward."""
    table = SSA_QX_MALE if sex == "male" else SSA_QX_FEMALE
    return table[age:] if age < len(table) else [1.0]


def get_baseline_life_expectancy(age, sex):
    """Get SSA baseline remaining life expectancy at given age."""
    table = SSA_EX_MALE if sex == "male" else SSA_EX_FEMALE
    if age < len(table):
        return table[age]
    return 0.5


def life_expectancy_from_qx(qx_curve, start_age):
    """
    Compute life expectancy from a modified q(x) curve.

    Core actuarial calculation:
    1. Start with 100,000 people at start_age
    2. Each year, some die based on q(x)
    3. Count total person-years lived
    4. Divide by 100,000 = expected remaining years
    """
    lx = 100000.0
    Tx = 0.0

    for qx in qx_curve:
        qx = min(qx, 1.0)
        dx = lx * qx
        Lx = lx - dx * 0.5  # uniform deaths within year
        Tx += Lx
        lx -= dx
        if lx < 0.5:
            break

    return Tx / 100000.0


def get_survival_curve(qx_curve, start_age):
    """Compute survival probabilities. Returns [(age, survival_prob), ...]."""
    survival = 1.0
    curve = [(start_age, 1.0)]

    for i, qx in enumerate(qx_curve):
        survival *= (1.0 - min(qx, 1.0))
        curve.append((start_age + i + 1, survival))
        if survival < 0.001:
            break

    return curve


baseline_le_male_20   = get_baseline_life_expectancy(20, "male")
baseline_le_female_20 = get_baseline_life_expectancy(20, "female")
print(f"\n✓ Section 5 complete — SSA 2022 life tables loaded")
print(f"  20-year-old male baseline:   {baseline_le_male_20} remaining years → dies ~{20 + baseline_le_male_20:.1f}")
print(f"  20-year-old female baseline: {baseline_le_female_20} remaining years → dies ~{20 + baseline_le_female_20:.1f}")

life_table_rows = []
for age in range(120):
    if age < len(SSA_QX_MALE):
        life_table_rows.append({
            "age": age, "sex": "male", "qx": SSA_QX_MALE[age],
            "ex": SSA_EX_MALE[age] if age < len(SSA_EX_MALE) else None,
            "source": "SSA_2022",
        })
    if age < len(SSA_QX_FEMALE):
        life_table_rows.append({
            "age": age, "sex": "female", "qx": SSA_QX_FEMALE[age],
            "ex": SSA_EX_FEMALE[age] if age < len(SSA_EX_FEMALE) else None,
            "source": "SSA_2022",
        })

baseline_df = pd.DataFrame(life_table_rows)
baseline_df.to_parquet(PROC / "baseline_life_tables.parquet", index=False)

## Section 6: Vitality Model

Models health as a **vitality score** that drifts downward over time. Death is predicted when vitality hits zero.

$$y(t) = y_0 + \zeta t + \sigma W(t) + \text{jumps}$$

| Parameter | Meaning |
|-----------|----------|
| $y_0$ | Initial vitality — how healthy you are right now |
| $\zeta$ | Drift rate — how fast vitality declines (always negative) |
| $\sigma$ | Volatility — trajectory uncertainty |
| $\lambda_{\text{jump}}$ | Jump hazard — probability of sudden health shocks |

Max swing anchored to Li et al. 2018 *Circulation* (5 vs 0 healthy factors ≈ 14-year gap).

In [ ]:
def features_to_vitality_params(f):
    """
    Convert numeric features into vitality model parameters.
    Coefficients are calibrated against published epidemiological hazard ratios.
    """
    # ── y0: Initial vitality ──
    y0 = 100.0
    y0 -= 0.6 * max(f["age"] - 40, 0)
    y0 -= 5.0 * (max(f["bmi"] - 25, 0) / 5.0)
    y0 += 3.0 * f["grandparent_longevity_score"]
    y0 -= 4.0 * f["bp_score"]
    y0 -= 4.0 * f["ldl_score"]
    y0 -= 5.0 * f["glucose_score"]
    y0 -= 5.0 * f["chronic_disease_score"]
    y0 -= 3.0 * f["overweight_score"]
    y0  = max(y0, 10.0)

    # ── ζ: Drift rate (always negative) ──
    zeta = -1.0
    # Good factors slow the decline
    zeta += 0.05 * f["fruitveg_score"]
    zeta += 0.08 * f["cardio_score"]
    zeta += 0.05 * f["weights_score"]
    zeta += 0.03 * f["stretch_score"]
    zeta += 0.05 * f["sleep_duration_score"]
    zeta += 0.03 * f["water_score"]
    zeta += 0.02 * f["checkup_score"]
    zeta += 0.02 * f["screening_score"]
    zeta += 0.03 * f["community_time_score"]
    zeta += 0.02 * f["relationship_score"]
    zeta += 0.02 * f["support_score"]
    # Bad factors accelerate decline
    zeta -= 0.08 * f["processed_food_score"]
    zeta -= 0.07 * f["sugar_score"]
    zeta -= 0.08 * f["sitting_score"]
    zeta -= 0.08 * f["sleep_disturbance_score"]
    zeta -= 0.10 * f["bp_score"]
    zeta -= 0.10 * f["ldl_score"]
    zeta -= 0.12 * f["glucose_score"]
    zeta -= 0.12 * f["chronic_disease_score"]
    zeta -= 0.06 * f["stress_score"]
    zeta -= 0.05 * f["mental_health_score"]
    zeta = min(zeta, -0.05)  # must stay negative

    # ── σ: Volatility ──
    sigma  = 1.0
    sigma += 0.2 * f["stress_score"]
    sigma -= 0.1 * f["support_score"]
    sigma  = max(0.5, sigma)

    # ── λ: Jump hazard ──
    lambda_jump  = 0.01
    lambda_jump += 0.02 * f["alcohol_score"]
    lambda_jump += 0.03 * f["nicotine_score"]
    lambda_jump -= 0.01 * f["screening_score"]
    lambda_jump -= 0.005 * f["checkup_score"]
    lambda_jump  = max(0.0, lambda_jump)

    return {
        "y0": round(y0, 2),
        "zeta": round(zeta, 4),
        "sigma": round(sigma, 2),
        "lambda_jump": round(lambda_jump, 4),
    }


def vitality_to_prediction(params, current_age, sex):
    """
    Convert vitality parameters into a life expectancy prediction.

    Uses SSA baseline as the anchor and shifts it by a bounded amount
    derived from the user's lifestyle scores. Maximum swing: ±12–15 years
    (Li et al. 2018 Circulation: 14-year gap between 0 vs 5 healthy factors).
    """
    y0   = params["y0"]
    zeta = params["zeta"]
    lam  = params["lambda_jump"]

    baseline_remaining  = get_baseline_life_expectancy(current_age, sex)
    baseline_death_age  = current_age + baseline_remaining

    AVERAGE_Y0   = 78.0
    AVERAGE_ZETA = -1.0

    y0_delta       = (y0 - AVERAGE_Y0) / AVERAGE_Y0
    zeta_delta     = (AVERAGE_ZETA - zeta) / abs(AVERAGE_ZETA)
    lifestyle_score = 0.4 * y0_delta - 0.6 * zeta_delta

    MAX_BONUS   = 12.0
    MAX_PENALTY = 15.0

    if lifestyle_score >= 0:
        year_adjustment = MAX_BONUS * (1 - math.exp(-1.5 * lifestyle_score))
    else:
        year_adjustment = -MAX_PENALTY * (1 - math.exp(1.2 * lifestyle_score))

    if lam > 0.03:
        year_adjustment -= min((lam - 0.01) * 30, 5.0)

    predicted_death_age = clamp(baseline_death_age + year_adjustment, current_age + 1, 97)
    remaining_years     = predicted_death_age - current_age
    death_date          = datetime.now() + timedelta(days=remaining_years * 365.25)

    return {
        "vitality_params":      params,
        "lifestyle_score":      round(lifestyle_score, 3),
        "year_adjustment":      round(year_adjustment, 1),
        "remaining_years":      round(remaining_years, 1),
        "predicted_death_age":  round(predicted_death_age, 1),
        "predicted_death_date": death_date.strftime("%B %d, %Y"),
        "baseline_death_age":   round(baseline_death_age, 1),
        "years_vs_baseline":    round(year_adjustment, 1),
    }


sample_params     = features_to_vitality_params(sample_features)
sample_prediction = vitality_to_prediction(
    sample_params,
    current_age=sample_features["age"],
    sex="male" if sample_features["sex_male"] else "female",
)

print(f"\n✓ Section 6 complete — Vitality model working")
print(f"  {sample_features['first_name']}'s vitality parameters:")
print(f"    y0  (initial vitality): {sample_params['y0']}")
print(f"    ζ   (drift rate):       {sample_params['zeta']}")
print(f"    σ   (volatility):       {sample_params['sigma']}")
print(f"    λ   (jump hazard):      {sample_params['lambda_jump']}")
print(f"  Prediction:")
print(f"    Predicted death age:    {sample_prediction['predicted_death_age']}")
print(f"    Predicted death date:   {sample_prediction['predicted_death_date']}")
print(f"    SSA baseline death:     {sample_prediction['baseline_death_age']}")
print(f"    Difference:             {sample_prediction['years_vs_baseline']:+.1f} years")

## Section 7: Cox Proportional Hazards (NHANES)

A real statistical model trained on NHANES mortality data (134,000+ people, up to 30 years follow-up, linked to death records).

**How it works:**
1. Download NHANES harmonized dataset
2. Merge with linked mortality file
3. Fit Cox PH model
4. Extract hazard ratios → multiply baseline `q(x)`

> **Prerequisite:** Download NHANES data to `/content/raw/` before running the training cells.

In [ ]:
def load_nhanes_mortality(mortality_path, questionnaire_path):
    """
    Load and merge NHANES mortality + questionnaire data.

    Mortality file columns:
      - SEQN: participant ID
      - mortstat: 0 = alive, 1 = dead at end of follow-up
      - permth_exm: person-months from exam to death or censoring
    """
    mort  = pd.read_csv(mortality_path)
    quest = pd.read_csv(questionnaire_path)
    df    = mort.merge(quest, on="SEQN", how="inner")
    df    = df[df["permth_exm"] > 0].copy()
    return df


def fit_cox_model(df, duration_col="permth_exm", event_col="mortstat", feature_cols=None):
    """Fit a Cox PH model on NHANES data. Returns the fitted model."""
    from lifelines import CoxPHFitter

    if feature_cols is None:
        feature_cols = [c for c in df.columns if c not in [duration_col, event_col, "SEQN"]]

    model_df = df[[duration_col, event_col] + feature_cols].dropna()
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(model_df, duration_col=duration_col, event_col=event_col)
    return cph


def apply_cox_hazard_ratio(baseline_qx, hazard_ratio, start_age):
    """
    Modify baseline mortality curve using a Cox hazard ratio.

    Formula: q_adjusted(x) = 1 - (1 - q_baseline(x))^HR

    Age-attenuation: HR effect weakens past age 60 because at very old
    ages mortality converges across risk groups.
    """
    adjusted = []
    for i, qx in enumerate(baseline_qx):
        age = start_age + i
        if age < 60:
            effective_hr = hazard_ratio
        elif age > 95:
            effective_hr = 1.0 + (hazard_ratio - 1.0) * 0.15
        else:
            t = (age - 60) / 35.0
            effective_hr = 1.0 + (hazard_ratio - 1.0) * (1.0 - t * 0.85)

        adjusted_qx = clamp(1.0 - (1.0 - qx) ** effective_hr, 0.0, 1.0)
        adjusted.append(adjusted_qx)
    return adjusted


# ── Training (uncomment after downloading NHANES data) ──
# nhanes_df = load_nhanes_mortality(
#     RAW / "nhanes_mortality.csv",
#     RAW / "nhanes_questionnaire.csv",
# )
# cph = fit_cox_model(nhanes_df)
# cph.summary.to_csv(OUT / "cox_coefficients.csv")
# print(f"Concordance index: {cph.concordance_index_:.3f}")
# print(cph.summary[["coef", "exp(coef)", "p"]])

print("\n✓ Section 7 complete — Cox PH framework ready")
print("  (Download NHANES data to /content/raw/ to train the real model)")

## Section 8: Paper Ingestion Pipeline (PubMed / PMC)

Automatically finds and downloads longevity research papers using NCBI's free E-utilities API:
- **ESearch** — find papers by keyword
- **ESummary** — metadata (title, authors, journal, year)
- **EFetch** — abstracts
- **PMC BioC API** — full text (open access only)

Get a free NCBI API key at https://www.ncbi.nlm.nih.gov/account/settings/ (raises rate limit from 3 → 10 req/s).

In [ ]:
import requests
import time

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def esearch(query, retmax=100, db="pubmed"):
    """Search PubMed; returns list of PMIDs."""
    params = {
        "db": db, "term": query, "retmode": "json",
        "retmax": retmax, "email": CONFIG["email"],
    }
    if CONFIG["ncbi_api_key"]:
        params["api_key"] = CONFIG["ncbi_api_key"]
    r = requests.get(f"{BASE_URL}/esearch.fcgi", params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])


def esummary(pmids, db="pubmed"):
    """Get metadata (title, journal, year, DOI, PMCID) for a list of PMIDs."""
    if not pmids:
        return {}
    params = {
        "db": db, "id": ",".join(pmids[:200]),
        "retmode": "json", "email": CONFIG["email"],
    }
    if CONFIG["ncbi_api_key"]:
        params["api_key"] = CONFIG["ncbi_api_key"]
    r = requests.get(f"{BASE_URL}/esummary.fcgi", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    results = {}
    for pmid in pmids:
        entry = data.get("result", {}).get(pmid, {})
        results[pmid] = {
            "pmid":    pmid,
            "title":   entry.get("title", ""),
            "journal": entry.get("fulljournalname", entry.get("source", "")),
            "year":    entry.get("pubdate", "")[:4],
            "doi":     next((a["value"] for a in entry.get("articleids", []) if a["idtype"] == "doi"), ""),
            "pmcid":   next((a["value"] for a in entry.get("articleids", []) if a["idtype"] == "pmc"), ""),
        }
    return results


def efetch_abstracts(pmids, db="pubmed"):
    """Fetch abstracts for a list of PMIDs as plain text."""
    if not pmids:
        return {}
    params = {
        "db": db, "id": ",".join(pmids[:200]),
        "rettype": "abstract", "retmode": "text",
        "email": CONFIG["email"],
    }
    if CONFIG["ncbi_api_key"]:
        params["api_key"] = CONFIG["ncbi_api_key"]
    r = requests.get(f"{BASE_URL}/efetch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.text


def fetch_pmc_fulltext(pmcid):
    """Fetch full text of an open-access paper via the PMC BioC API."""
    url = (
        f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi"
        f"/BioC_json/{pmcid}/unicode"
    )
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            data = r.json()
            texts = []
            for doc in data.get("documents", []):
                for passage in doc.get("passages", []):
                    texts.append(passage.get("text", ""))
            return "\n\n".join(texts)
    except Exception:
        pass
    return None

In [ ]:
QUERY_BANK = {
    "smoking": [
        '"smoking"[MeSH] AND "mortality"[MeSH] AND cohort[tiab]',
        '"smoking cessation"[MeSH] AND all-cause mortality',
        '(nicotine OR vaping OR e-cigarette) AND mortality AND cohort',
    ],
    "alcohol": [
        '"alcohol drinking"[MeSH] AND "mortality"[MeSH]',
        'alcohol consumption all-cause mortality meta-analysis',
    ],
    "diet": [
        'fruit vegetable intake all-cause mortality meta-analysis',
        'ultra-processed food all-cause mortality',
        'sugar-sweetened beverages mortality cohort',
    ],
    "exercise": [
        '"motor activity"[MeSH] AND mortality',
        'physical activity all-cause mortality meta-analysis',
        'sedentary behavior all-cause mortality',
    ],
    "sleep": [
        'sleep duration all-cause mortality meta-analysis',
        'insomnia mortality cohort adults',
    ],
    "bmi": [
        '"body mass index"[MeSH] AND mortality',
        'obesity all-cause mortality cohort adults',
    ],
    "blood_pressure": [
        '"hypertension"[MeSH] AND mortality',
        'systolic blood pressure mortality hazard ratio',
    ],
    "cholesterol": [
        'LDL cholesterol all-cause mortality cohort',
        'dyslipidemia mortality meta-analysis',
    ],
    "glucose": [
        'fasting glucose all-cause mortality cohort',
        'diabetes mortality NHANES cohort',
    ],
    "chronic_disease": [
        'multimorbidity all-cause mortality cohort',
        'chronic conditions life expectancy cohort',
    ],
    "mental_health": [
        '"stress, psychological"[MeSH] AND mortality',
        '"depression"[MeSH] AND all-cause mortality',
    ],
    "social": [
        '"social support"[MeSH] AND mortality',
        'loneliness all-cause mortality meta-analysis',
    ],
    "healthcare": [
        'preventive health services mortality cohort',
        'cancer screening mortality cohort',
    ],
    "genetics": [
        'parental longevity offspring mortality cohort',
        'familial longevity cohort study',
    ],
    "methodology": [
        '"Cox proportional hazards"[tiab] AND mortality prediction',
        'vitality model mortality first hitting time',
    ],
}

with open(PROC / "paper_query_bank.json", "w") as f:
    json.dump(QUERY_BANK, f, indent=2)


def run_ingestion(factor_areas=None, papers_per_query=20):
    """Run the full paper ingestion pipeline for selected factor areas."""
    if factor_areas is None:
        factor_areas = list(QUERY_BANK.keys())

    all_papers  = []
    seen_pmids  = set()

    for area in factor_areas:
        for query in QUERY_BANK.get(area, []):
            print(f"  Searching [{area}]: {query[:60]}...")
            pmids     = esearch(query, retmax=papers_per_query)
            new_pmids = [p for p in pmids if p not in seen_pmids]

            if new_pmids:
                metadata = esummary(new_pmids)
                for pmid, meta in metadata.items():
                    meta["factor_area"] = area
                    all_papers.append(meta)
                    seen_pmids.add(pmid)

            time.sleep(0.35)  # respect NCBI rate limit

    df = pd.DataFrame(all_papers)
    df.to_csv(PROC / "paper_metadata.csv", index=False)
    print(f"\n  Total unique papers found: {len(df)}")
    return df


# Uncomment to run (takes a few minutes):
# paper_df = run_ingestion(factor_areas=["smoking", "exercise", "sleep"], papers_per_query=20)

print(f"\n✓ Section 8 complete — paper ingestion pipeline ready")
print(f"  {len(QUERY_BANK)} factor areas, {sum(len(v) for v in QUERY_BANK.values())} queries defined")

## Section 9: Chunking, Embeddings & Vector DB

Papers are too long to feed to an LLM directly. Pipeline:
1. Split papers into ~500-token chunks (≈ 2200 chars with 250-char overlap)
2. Embed each chunk with `sentence-transformers/all-MiniLM-L6-v2` (384-dim, CPU-friendly)
3. Store in **ChromaDB** for fast similarity search

When a user has high nicotine score, we retrieve the most relevant chunks about smoking + mortality to power the explanation.

In [ ]:
def simple_chunk(text, chunk_size=2200, overlap=250):
    """
    Split text into overlapping chunks (~500 tokens each).
    Overlap prevents cutting off context at boundaries.
    """
    chunks = []
    start  = 0
    while start < len(text):
        end   = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def process_papers_to_chunks(paper_df, fetch_fulltext=False):
    """Convert paper metadata into chunks ready for embedding."""
    all_chunks = []
    for _, row in paper_df.iterrows():
        pmid = row["pmid"]
        text = row.get("abstract", row.get("title", ""))

        if fetch_fulltext and row.get("pmcid"):
            fulltext = fetch_pmc_fulltext(row["pmcid"])
            if fulltext:
                text = fulltext

        if not text or len(text) < 50:
            continue

        for i, chunk_text in enumerate(simple_chunk(text)):
            all_chunks.append({
                "paper_id":    pmid,
                "chunk_id":    f"{pmid}_chunk_{i}",
                "text":        chunk_text,
                "title":       row.get("title", ""),
                "year":        row.get("year", ""),
                "pmid":        pmid,
                "pmcid":       row.get("pmcid", ""),
                "factor_area": row.get("factor_area", ""),
                "journal":     row.get("journal", ""),
            })
    return all_chunks


def build_vector_db(chunks, collection_name="longevity_papers"):
    """
    Embed chunks with all-MiniLM-L6-v2 and store in ChromaDB.
    Upgrade to nomic-embed-text for higher retrieval quality.
    """
    from sentence_transformers import SentenceTransformer
    import chromadb

    print("  Loading embedding model...")
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    client   = chromadb.PersistentClient(path=str(PROC / "chromadb"))

    try:
        client.delete_collection(collection_name)
    except Exception:
        pass

    collection = client.create_collection(
        name=collection_name,
        metadata={"description": "Longevity research paper chunks"},
    )

    batch_size = 64
    for i in range(0, len(chunks), batch_size):
        batch     = chunks[i:i + batch_size]
        texts     = [c["text"] for c in batch]
        ids       = [c["chunk_id"] for c in batch]
        metadatas = [{k: v for k, v in c.items() if k != "text"} for c in batch]
        embeddings = embedder.encode(texts).tolist()
        collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
        print(f"  Embedded {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")

    return collection, embedder


# Uncomment after running Section 8's ingestion:
# chunks = process_papers_to_chunks(paper_df, fetch_fulltext=True)
# collection, embedder = build_vector_db(chunks)
# with open(PROC / "paper_chunks.jsonl", "w") as f:
#     for chunk in chunks:
#         f.write(json.dumps(chunk) + "\n")

print("\n✓ Section 9 complete — chunking & vector DB framework ready")

## Section 10: Retrieval & Explanation Assembly

Given a user's results:
1. Identify their highest-risk factors
2. Generate targeted PubMed-style queries
3. Retrieve relevant evidence from the vector DB
4. Assemble an LLM prompt that explains the prediction

> **Important:** The LLM only *explains* the death date — it never invents or changes the number computed by the statistical model.

In [ ]:
def identify_risk_factors(features, threshold=2):
    """
    Find the user's most concerning risk factors.
    Returns list of (factor_name, score, risk_level) sorted by severity.
    """
    bad_factors = {
        "processed_food_score":  "processed food consumption",
        "sugar_score":           "sugar intake",
        "sitting_score":         "sedentary time",
        "sleep_disturbance_score": "sleep problems",
        "alcohol_score":         "alcohol consumption",
        "nicotine_score":        "nicotine/tobacco use",
        "stress_score":          "chronic stress",
        "mental_health_score":   "mental health impact",
        "bp_score":              "blood pressure",
        "ldl_score":             "LDL cholesterol",
        "glucose_score":         "blood glucose",
        "chronic_disease_score": "chronic disease",
        "overweight_score":      "weight/BMI",
    }
    good_factors = {
        "fruitveg_score":        "fruit and vegetable intake",
        "cardio_score":          "cardiovascular exercise",
        "weights_score":         "strength training",
        "sleep_duration_score":  "sleep duration",
        "community_time_score":  "social connection",
        "support_score":         "social support",
        "screening_score":       "cancer screenings",
        "checkup_score":         "regular checkups",
    }

    risks = []
    for key, label in bad_factors.items():
        if features.get(key, 0) >= threshold:
            risks.append((label, features[key], "high_risk"))
    for key, label in good_factors.items():
        score = features.get(key, 0)
        if score <= 1:
            risks.append((label, 3 - score, "needs_improvement"))

    risks.sort(key=lambda x: x[1], reverse=True)
    return risks


def user_to_retrieval_queries(features):
    """Generate targeted search queries based on the user's risk profile."""
    queries = []
    if features.get("nicotine_score", 0) >= 2:
        queries.append("current smoking all-cause mortality cohort adults")
    if features.get("alcohol_score", 0) >= 2:
        queries.append("heavy alcohol consumption mortality risk adults")
    if features.get("sleep_disturbance_score", 0) >= 2:
        queries.append("sleep disturbance mortality meta-analysis adults")
    if features.get("sleep_duration_score", 0) <= 1:
        queries.append("short sleep duration mortality risk")
    if features.get("ldl_score", 0) >= 2:
        queries.append("LDL cholesterol all-cause mortality cohort")
    if features.get("bp_score", 0) >= 2:
        queries.append("hypertension blood pressure mortality hazard ratio")
    if features.get("glucose_score", 0) >= 2:
        queries.append("diabetes fasting glucose mortality risk")
    if features.get("sitting_score", 0) >= 2:
        queries.append("sedentary behavior prolonged sitting mortality")
    if features.get("cardio_score", 0) <= 1:
        queries.append("physical inactivity mortality risk exercise benefits")
    if features.get("stress_score", 0) >= 2:
        queries.append("chronic psychological stress mortality cortisol")
    if features.get("processed_food_score", 0) >= 2:
        queries.append("ultra-processed food consumption mortality risk")
    if not queries:
        queries.append("lifestyle factors all-cause mortality meta-analysis")
    return queries


def retrieve_evidence(queries, collection, embedder, top_k=5):
    """Search vector DB for evidence. Returns de-duplicated chunks sorted by relevance."""
    all_results  = []
    seen_papers  = set()

    for query in queries:
        results = collection.query(
            query_embeddings=embedder.encode([query]).tolist(),
            n_results=top_k,
        )
        for i, doc_id in enumerate(results["ids"][0]):
            paper_id = results["metadatas"][0][i].get("paper_id", "")
            if paper_id not in seen_papers:
                seen_papers.add(paper_id)
                all_results.append({
                    "chunk_id": doc_id,
                    "text":     results["documents"][0][i],
                    "metadata": results["metadatas"][0][i],
                    "distance": results["distances"][0][i] if "distances" in results else 0,
                })

    all_results.sort(key=lambda x: x["distance"])
    return all_results[:15]


def build_explanation_payload(raw_answers, features, prediction, retrieved_chunks):
    """Assemble the full context dict for the LLM explanation."""
    risks       = identify_risk_factors(features)
    clinical    = {"blood pressure", "LDL cholesterol", "blood glucose", "chronic disease"}
    actionable  = [r for r in risks if r[0] not in clinical]
    non_action  = [r for r in risks if r[0] in clinical]

    return {
        "user_profile": {
            "name": raw_answers.get("first_name", "User"),
            "age":  features["age"],
            "sex":  "male" if features["sex_male"] else "female",
            "bmi":  features["bmi"],
        },
        "prediction": prediction,
        "risk_factors": {
            "actionable": [(r[0], r[1]) for r in actionable],
            "clinical":   [(r[0], r[1]) for r in non_action],
        },
        "evidence_chunks": [
            {
                "text":   c["text"][:500],
                "pmid":   c["metadata"].get("pmid", ""),
                "title":  c["metadata"].get("title", ""),
                "factor": c["metadata"].get("factor_area", ""),
            }
            for c in retrieved_chunks
        ],
        "instructions": (
            "You are a longevity health advisor. Explain the user's life expectancy prediction. "
            "NEVER change or invent the death date — only explain the existing numbers. "
            "Cite evidence by PMID when available. "
            "Separate actionable factors (diet, exercise, sleep) from clinical factors (BP, cholesterol). "
            "Be encouraging but honest. Note uncertainty in all predictions."
        ),
    }


sample_risks   = identify_risk_factors(sample_features)
sample_queries = user_to_retrieval_queries(sample_features)

print(f"\n✓ Section 10 complete — retrieval & explanation framework ready")
print(f"  {sample_features['first_name']}'s risk factors:")
for label, score, level in sample_risks[:5]:
    icon = "⚠️" if level == "high_risk" else "📈"
    print(f"    {icon} {label}: {score}/3 ({level})")
print(f"  Generated {len(sample_queries)} retrieval queries")

## Section 11: Evaluation & Export

Validates the model with three archetypal profiles:
- **Healthy Hana** — best-case lifestyle, 25F
- **Average Alex** — middle-of-the-road, 45M
- **Risky Rick** — worst-case lifestyle, 55M

**Assertion:** `predicted_death_age(healthy) > average > high_risk`

In [ ]:
def create_test_profiles():
    """Generate test profiles spanning the full risk spectrum."""
    base = dict(SAMPLE_USER)

    healthy = dict(base)
    healthy.update({
        "first_name": "Healthy Hana", "age": 25, "sex": "female",
        "diet_fruits_veggies": "five_or_more_servings_a_day",
        "diet_processed_foods": "rarely_or_never",
        "diet_sugar": "none",
        "exercise_cardio": "more_than_300_minutes",
        "exercise_weights": "more_than_two_days_per_week",
        "sleep_duration": "five_or_more_nights_per_week",
        "sleep_trouble": "never",
        "nicotine": "never_used",
        "alcohol": "dont_drink",
        "stress": "rarely_or_never",
        "mental_health_impact": "not_at_all",
        "blood_pressure": "below_120_80_normal",
        "glucose": "below_100_normal",
        "chronic_disease": "no",
        "grandparents_max_age": "90_plus",
    })

    average = dict(base)
    average.update({
        "first_name": "Average Alex", "age": 45, "sex": "male",
        "diet_fruits_veggies": "several_times_a_week",
        "exercise_cardio": "less_than_150_minutes",
        "exercise_sitting": "four_to_eight_hours",
        "sleep_duration": "three_to_four_nights_per_week",
        "alcohol": "1_7_drinks_per_week",
        "stress": "occasionally",
        "grandparents_max_age": "70_79",
    })

    high_risk = dict(base)
    high_risk.update({
        "first_name": "Risky Rick", "age": 55, "sex": "male",
        "diet_fruits_veggies": "rarely_or_never",
        "diet_processed_foods": "daily",
        "diet_sugar": "sweets_several_times_a_day",
        "exercise_cardio": "rarely_or_never",
        "exercise_sitting": "more_than_8_hours",
        "sleep_duration": "never",
        "sleep_trouble": "five_or_more_nights_per_week",
        "nicotine": "current_daily_user",
        "alcohol": "15_or_more_drinks_per_week",
        "stress": "almost_all_the_time",
        "mental_health_impact": "severely",
        "blood_pressure": "140_90_or_higher_high",
        "glucose": "126_or_higher_diabetes",
        "chronic_disease": "yes",
        "grandparents_max_age": "under_70",
        "overweight": "obese",
    })

    return {"healthy": healthy, "average": average, "high_risk": high_risk}


print("\n" + "=" * 60)
print("EVALUATION: Test Profile Results")
print("=" * 60)

profiles = create_test_profiles()
results  = []

for name, answers in profiles.items():
    features   = questionnaire_to_features(answers)
    params     = features_to_vitality_params(features)
    sex        = "male" if features["sex_male"] else "female"
    prediction = vitality_to_prediction(params, features["age"], sex)

    results.append({
        "profile":             name,
        "name":                features["first_name"],
        "age":                 features["age"],
        "sex":                 sex,
        "bmi":                 features["bmi"],
        "vitality_y0":         params["y0"],
        "vitality_zeta":       params["zeta"],
        "predicted_death_age": prediction["predicted_death_age"],
        "baseline_death_age":  prediction["baseline_death_age"],
        "delta_years":         prediction["years_vs_baseline"],
        "death_date":          prediction["predicted_death_date"],
    })

    print(f"\n  {features['first_name']} ({name}, age {features['age']}, {sex}):")
    print(f"    Vitality: y0={params['y0']}, ζ={params['zeta']}")
    print(f"    Predicted death age: {prediction['predicted_death_age']}")
    print(f"    vs. baseline:        {prediction['baseline_death_age']} ({prediction['years_vs_baseline']:+.1f} years)")

death_ages = {r["profile"]: r["predicted_death_age"] for r in results}
assert death_ages["healthy"]  > death_ages["average"],   "Healthy should outlive average!"
assert death_ages["average"]  > death_ages["high_risk"], "Average should outlive high risk!"
print("\n  ✓ Risk ordering validated: healthy > average > high_risk")

results_df = pd.DataFrame(results)
results_df.to_csv(OUT / "evaluation_results.csv", index=False)

with open(OUT / "sample_profiles.json", "w") as f:
    json.dump(profiles, f, indent=2, default=str)

print(f"\n✓ Section 11 complete — all evaluation passed")
print(f"\n{'=' * 60}")
print("NOTEBOOK COMPLETE")
print(f"{'=' * 60}")
print(f"Files saved to:")
print(f"  {PROC}/questionnaire_schema.json")
print(f"  {PROC}/features_definition.json")
print(f"  {PROC}/paper_query_bank.json")
print(f"  {PROC}/baseline_life_tables.parquet")
print(f"  {OUT}/feature_matrix.csv")
print(f"  {OUT}/evaluation_results.csv")
print(f"  {OUT}/sample_profiles.json")

## Section 12: Data Visualization

Six plots that give an intuition for how every layer of the engine works:

1. **SSA q(x) Mortality Rates** — raw annual death probability by age & sex
2. **Survival Curves** — probability of still being alive at each age
3. **Feature Scores — Sample User** — radar / bar breakdown of the 30 lifestyle scores
4. **Vitality Trajectory** — y(t) drift curves for all three test profiles
5. **Year Adjustment Distribution** — how far different lifestyle combinations push the prediction away from the SSA baseline
6. **Profile Comparison** — predicted vs. baseline death age for each test profile

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#cccccc",
    "axes.titlecolor":  "#ffffff",
    "xtick.color":      "#888",
    "ytick.color":      "#888",
    "grid.color":       "#2a2a2a",
    "grid.linewidth":   0.8,
    "text.color":       "#cccccc",
    "font.family":      "monospace",
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.labelsize":   10,
})

BLUE   = "#4da6ff"
PINK   = "#ff6b9d"
GREEN  = "#50fa7b"
YELLOW = "#f1fa8c"
ORANGE = "#ffb86c"
RED    = "#ff5555"
PURPLE = "#bd93f9"
CYAN   = "#8be9fd"

print("✓ Matplotlib configured — dark theme ready")

In [ ]:
# ── Plot 1: SSA Annual Mortality Rates q(x) ──

ages = np.arange(len(SSA_QX_MALE))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SSA 2022 Period Life Table — Annual Mortality Rate q(x)", fontsize=13, color="white", y=1.01)

# Linear scale
ax = axes[0]
ax.plot(ages, SSA_QX_MALE,   color=BLUE,  lw=2, label="Male")
ax.plot(ages, SSA_QX_FEMALE, color=PINK,  lw=2, label="Female")
ax.fill_between(ages, SSA_QX_MALE,   alpha=0.12, color=BLUE)
ax.fill_between(ages, SSA_QX_FEMALE, alpha=0.12, color=PINK)
ax.axvline(65, color=YELLOW, lw=1, ls="--", alpha=0.6, label="Age 65")
ax.set_xlabel("Age")
ax.set_ylabel("Probability of death within 1 year")
ax.set_title("Linear scale")
ax.legend(facecolor="#222", edgecolor="#555")
ax.grid(True)

# Log scale — reveals the Gompertz exponential shape
ax = axes[1]
ax.semilogy(ages, SSA_QX_MALE,   color=BLUE, lw=2, label="Male")
ax.semilogy(ages, SSA_QX_FEMALE, color=PINK, lw=2, label="Female")
ax.axvline(65, color=YELLOW, lw=1, ls="--", alpha=0.6, label="Age 65")
ax.set_xlabel("Age")
ax.set_ylabel("q(x)  [log scale]")
ax.set_title("Log scale — Gompertz shape")
ax.legend(facecolor="#222", edgecolor="#555")
ax.grid(True, which="both")

plt.tight_layout()
plt.savefig(OUT / "plot_qx_mortality.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 1 saved")

In [ ]:
# ── Plot 2: Survival Curves for Test Profiles ──

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Survival Curves — SSA Baseline vs. Lifestyle-Adjusted", fontsize=13, color="white")

profile_styles = {
    "healthy":   (GREEN,  "Healthy Hana (25F)",   "solid"),
    "average":   (YELLOW, "Average Alex (45M)",   "solid"),
    "high_risk": (RED,    "Risky Rick (55M)",     "solid"),
}

for profile_name, answers in profiles.items():
    feats  = questionnaire_to_features(answers)
    params = features_to_vitality_params(feats)
    sex    = "male" if feats["sex_male"] else "female"
    age    = feats["age"]
    pred   = vitality_to_prediction(params, age, sex)

    color, label, ls = profile_styles[profile_name]

    # SSA baseline survival curve
    baseline_qx = get_baseline_qx(age, sex)
    baseline_curve = get_survival_curve(baseline_qx, age)
    b_ages  = [p[0] for p in baseline_curve]
    b_surv  = [p[1] for p in baseline_curve]
    ax.plot(b_ages, b_surv, color=color, lw=1, ls="dashed", alpha=0.45)

    # Lifestyle-shifted curve (scale baseline by year adjustment)
    shift = pred["year_adjustment"]
    adj_ages = [a + shift for a in b_ages]
    ax.plot(adj_ages, b_surv, color=color, lw=2.5, ls=ls, label=label)

    # Mark predicted death age
    ax.axvline(pred["predicted_death_age"], color=color, lw=1, ls=":", alpha=0.7)
    ax.annotate(
        f"{pred['predicted_death_age']:.0f}",
        xy=(pred["predicted_death_age"], 0.05),
        color=color, fontsize=9, ha="center",
    )

ax.axhline(0.5, color="#555", lw=1, ls="--", alpha=0.6)
ax.text(20, 0.52, "50% survival", color="#888", fontsize=8)
ax.set_xlabel("Age")
ax.set_ylabel("Probability of survival")
ax.set_xlim(15, 105)
ax.set_ylim(-0.02, 1.05)
ax.legend(facecolor="#222", edgecolor="#555", title="Solid = adjusted | Dashed = SSA baseline")
ax.grid(True)

plt.tight_layout()
plt.savefig(OUT / "plot_survival_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 2 saved")

In [ ]:
# ── Plot 3: Sample User Feature Score Breakdown ──

score_labels = [
    ("fruitveg_score",            "Fruit & Veg",      GREEN),
    ("water_score",               "Water",            CYAN),
    ("processed_food_score",      "Processed Food",   RED),
    ("sugar_score",               "Sugar",            ORANGE),
    ("cardio_score",              "Cardio",           GREEN),
    ("weights_score",             "Weights",          CYAN),
    ("stretch_score",             "Mobility",         BLUE),
    ("sitting_score",             "Sitting",          RED),
    ("activity_tracking_score",   "Tracking",         PURPLE),
    ("sleep_duration_score",      "Sleep Duration",   GREEN),
    ("sleep_disturbance_score",   "Sleep Trouble",    RED),
    ("community_time_score",      "Community",        YELLOW),
    ("relationship_score",        "Relationship",     PINK),
    ("support_score",             "Social Support",   CYAN),
    ("income_score",              "Income",           PURPLE),
    ("bloodwork_score",           "Bloodwork",        BLUE),
    ("alcohol_score",             "Alcohol",          ORANGE),
    ("nicotine_score",            "Nicotine",         RED),
    ("stress_score",              "Stress",           RED),
    ("mental_health_score",       "Mental Health",    ORANGE),
    ("checkup_score",             "Checkups",         GREEN),
    ("screening_score",           "Screenings",       GREEN),
    ("grandparent_longevity_score","Grandparents",    PURPLE),
    ("overweight_score",          "Overweight",       ORANGE),
    ("bp_score",                  "Blood Pressure",   RED),
    ("ldl_score",                 "LDL",              ORANGE),
    ("glucose_score",             "Glucose",          YELLOW),
    ("chronic_disease_score",     "Chronic Disease",  RED),
]

keys   = [s[0] for s in score_labels]
names  = [s[1] for s in score_labels]
colors = [s[2] for s in score_labels]
vals   = [sample_features.get(k, 0) for k in keys]

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    f"Feature Score Breakdown — {sample_features['first_name']} "
    f"(age {sample_features['age']}, BMI {sample_features['bmi']})",
    fontsize=13, color="white",
)

x = np.arange(len(names))
bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor="#333", linewidth=0.6)

# score label on each bar
for bar, v in zip(bars, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2, v + 0.04,
        str(v), ha="center", va="bottom", fontsize=7.5, color="#ccc",
    )

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Score (0–3)")
ax.set_ylim(0, 3.6)
ax.axhline(3, color="#555", lw=0.8, ls="--")
ax.text(len(names) - 0.5, 3.05, "max", color="#666", fontsize=8)
ax.grid(True, axis="y")

good_patch = mpatches.Patch(color=GREEN,  label="Good factor (higher = better)")
bad_patch  = mpatches.Patch(color=RED,    label="Risk factor (higher = worse)")
misc_patch = mpatches.Patch(color=PURPLE, label="Other")
ax.legend(handles=[good_patch, bad_patch, misc_patch], facecolor="#222", edgecolor="#555")

plt.tight_layout()
plt.savefig(OUT / "plot_feature_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 3 saved")

In [ ]:
# ── Plot 4: Vitality Trajectory y(t) for All Three Profiles ──
# y(t) = y0 + zeta * t   (deterministic drift, no noise for clarity)

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Vitality Trajectory  y(t) = y₀ + ζ·t", fontsize=13, color="white")

profile_colors = {"healthy": GREEN, "average": YELLOW, "high_risk": RED}

for profile_name, answers in profiles.items():
    feats  = questionnaire_to_features(answers)
    params = features_to_vitality_params(feats)
    age    = feats["age"]
    label  = profile_styles[profile_name][1]
    color  = profile_colors[profile_name]

    y0   = params["y0"]
    zeta = params["zeta"]

    t_max   = max(1, int(-y0 / zeta) + 5)  # years until vitality ~ 0
    t       = np.linspace(0, t_max, 400)
    yt      = y0 + zeta * t
    ages_t  = age + t

    # Only plot while alive
    mask = yt > 0
    ax.plot(ages_t[mask], yt[mask], color=color, lw=2.5, label=f"{label}  (y₀={y0}, ζ={zeta})")

    # Shade uncertainty band ±sigma
    sigma = params["sigma"]
    ax.fill_between(
        ages_t[mask],
        (yt - sigma * np.sqrt(t))[mask],
        (yt + sigma * np.sqrt(t))[mask],
        color=color, alpha=0.1,
    )

    # Mark where vitality hits 0
    t_death = -y0 / zeta
    ax.scatter([age + t_death], [0], color=color, s=60, zorder=5)

ax.axhline(0, color="#666", lw=1.5, ls="--", label="Vitality = 0 (predicted death)")
ax.set_xlabel("Age")
ax.set_ylabel("Vitality y(t)")
ax.set_xlim(15, 105)
ax.legend(facecolor="#222", edgecolor="#555", fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.savefig(OUT / "plot_vitality_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 4 saved")

In [ ]:
# ── Plot 5: Year-Adjustment Sensitivity ──
# How much does each individual score (0→3) move the predicted death age?

SENSITIVITY_FACTORS = [
    ("cardio_score",          "Cardio exercise",       "good"),
    ("sleep_duration_score",  "Sleep duration",        "good"),
    ("fruitveg_score",        "Fruit & Veg",           "good"),
    ("weights_score",         "Strength training",     "good"),
    ("community_time_score",  "Social connection",     "good"),
    ("nicotine_score",        "Nicotine use",          "bad"),
    ("chronic_disease_score", "Chronic disease",       "bad"),
    ("glucose_score",         "Blood glucose",         "bad"),
    ("bp_score",              "Blood pressure",        "bad"),
    ("sitting_score",         "Sedentary time",        "bad"),
    ("stress_score",          "Chronic stress",        "bad"),
    ("alcohol_score",         "Alcohol",               "bad"),
]

# Start from a neutral baseline (all scores = 1, average-ish)
neutral_answers = dict(SAMPLE_USER)
neutral_feats   = questionnaire_to_features(neutral_answers)
neutral_params  = features_to_vitality_params(neutral_feats)
neutral_pred    = vitality_to_prediction(neutral_params, neutral_feats["age"], "male")
baseline_age    = neutral_pred["predicted_death_age"]

factor_names, deltas, bar_colors = [], [], []

for feat_key, label, direction in SENSITIVITY_FACTORS:
    lo_feats  = dict(neutral_feats)
    hi_feats  = dict(neutral_feats)
    lo_feats[feat_key] = 0
    hi_feats[feat_key] = 3

    lo_pred = vitality_to_prediction(features_to_vitality_params(lo_feats), neutral_feats["age"], "male")
    hi_pred = vitality_to_prediction(features_to_vitality_params(hi_feats), neutral_feats["age"], "male")

    swing = hi_pred["predicted_death_age"] - lo_pred["predicted_death_age"]
    # For bad factors the sign flips (higher score = worse), re-orient so positive = healthier
    if direction == "bad":
        swing = -swing

    factor_names.append(label)
    deltas.append(swing)
    bar_colors.append(GREEN if swing > 0 else RED)

# Sort by magnitude
order   = np.argsort(np.abs(deltas))[::-1]
names_s = [factor_names[i] for i in order]
vals_s  = [deltas[i]       for i in order]
cols_s  = [bar_colors[i]   for i in order]

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle(
    "Sensitivity: Score 0→3 Swing in Predicted Death Age\n"
    "(positive = score 3 adds years vs score 0)",
    fontsize=11, color="white",
)

y_pos = np.arange(len(names_s))
bars  = ax.barh(y_pos, vals_s, color=cols_s, alpha=0.85, edgecolor="#333", height=0.6)

for bar, v in zip(bars, vals_s):
    ax.text(
        v + (0.05 if v >= 0 else -0.05),
        bar.get_y() + bar.get_height() / 2,
        f"{v:+.1f} yr", va="center",
        ha="left" if v >= 0 else "right",
        fontsize=9, color="#ccc",
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(names_s)
ax.axvline(0, color="#666", lw=1)
ax.set_xlabel("Year adjustment (relative to score=0)")
ax.grid(True, axis="x")

plt.tight_layout()
plt.savefig(OUT / "plot_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 5 saved")

In [ ]:
# ── Plot 6: Profile Comparison Dashboard ──

fig = plt.figure(figsize=(14, 8))
fig.suptitle("Profile Comparison Dashboard", fontsize=14, color="white", y=1.01)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

all_preds   = {}
all_feats   = {}
all_params  = {}

for name, answers in profiles.items():
    f = questionnaire_to_features(answers)
    p = features_to_vitality_params(f)
    s = "male" if f["sex_male"] else "female"
    d = vitality_to_prediction(p, f["age"], s)
    all_feats[name]  = f
    all_params[name] = p
    all_preds[name]  = d

pnames  = ["healthy", "average", "high_risk"]
plabels = ["Healthy\nHana", "Average\nAlex", "Risky\nRick"]
pcolors = [GREEN, YELLOW, RED]

# 6a — Predicted vs. Baseline death age
ax1 = fig.add_subplot(gs[0, 0])
x   = np.arange(3)
w   = 0.35
base_vals = [all_preds[n]["baseline_death_age"] for n in pnames]
pred_vals = [all_preds[n]["predicted_death_age"] for n in pnames]
ax1.bar(x - w/2, base_vals, w, label="SSA Baseline", color="#555", alpha=0.8)
for i, (n, pv, bv) in enumerate(zip(pnames, pred_vals, base_vals)):
    ax1.bar(i + w/2, pv, w, color=pcolors[i], alpha=0.85)
    ax1.annotate(
        f"{pv:.0f}",
        (i + w/2, pv), ha="center", va="bottom", fontsize=8.5, color=pcolors[i],
    )
ax1.set_xticks(x)
ax1.set_xticklabels(plabels, fontsize=8)
ax1.set_ylabel("Predicted death age")
ax1.set_title("Predicted vs. Baseline")
ax1.legend(facecolor="#222", edgecolor="#555", fontsize=7)
ax1.set_ylim(60, 100)
ax1.grid(True, axis="y")

# 6b — Vitality y0
ax2 = fig.add_subplot(gs[0, 1])
y0s = [all_params[n]["y0"] for n in pnames]
bars = ax2.bar(plabels, y0s, color=pcolors, alpha=0.85, edgecolor="#333")
for bar, v in zip(bars, y0s):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.5, f"{v}", ha="center", fontsize=9, color="#ccc")
ax2.set_ylabel("y₀ (initial vitality)")
ax2.set_title("Initial Vitality y₀")
ax2.grid(True, axis="y")

# 6c — Drift rate zeta
ax3 = fig.add_subplot(gs[0, 2])
zetas = [all_params[n]["zeta"] for n in pnames]
bars  = ax3.bar(plabels, zetas, color=pcolors, alpha=0.85, edgecolor="#333")
for bar, v in zip(bars, zetas):
    ax3.text(bar.get_x() + bar.get_width()/2, v - 0.005, f"{v:.3f}", ha="center", fontsize=9, color="#ccc", va="top")
ax3.set_ylabel("ζ (drift rate)")
ax3.set_title("Vitality Drift Rate ζ")
ax3.axhline(0, color="#666", lw=0.8)
ax3.grid(True, axis="y")

# 6d — Year adjustment waterfall
ax4 = fig.add_subplot(gs[1, 0])
adjustments = [all_preds[n]["year_adjustment"] for n in pnames]
cols = [GREEN if a > 0 else RED for a in adjustments]
bars = ax4.bar(plabels, adjustments, color=cols, alpha=0.85, edgecolor="#333")
for bar, v in zip(bars, adjustments):
    ax4.text(bar.get_x() + bar.get_width()/2,
             v + (0.1 if v >= 0 else -0.1),
             f"{v:+.1f}", ha="center",
             va="bottom" if v >= 0 else "top",
             fontsize=9, color="#ccc")
ax4.axhline(0, color="#666", lw=0.8)
ax4.set_ylabel("Years vs. SSA baseline")
ax4.set_title("Lifestyle Year Adjustment")
ax4.grid(True, axis="y")

# 6e — BMI comparison
ax5 = fig.add_subplot(gs[1, 1])
bmis  = [all_feats[n]["bmi"] for n in pnames]
bars  = ax5.bar(plabels, bmis, color=pcolors, alpha=0.85, edgecolor="#333")
for bar, v in zip(bars, bmis):
    ax5.text(bar.get_x() + bar.get_width()/2, v + 0.2, f"{v}", ha="center", fontsize=9, color="#ccc")
ax5.axhline(18.5, color=CYAN,   lw=1, ls="--", label="Underweight (18.5)")
ax5.axhline(25,   color=YELLOW, lw=1, ls="--", label="Overweight (25)")
ax5.axhline(30,   color=RED,    lw=1, ls="--", label="Obese (30)")
ax5.set_ylabel("BMI")
ax5.set_title("BMI")
ax5.legend(facecolor="#222", edgecolor="#555", fontsize=7)
ax5.grid(True, axis="y")

# 6f — Lifestyle score radar (spider chart)
ax6 = fig.add_subplot(gs[1, 2], polar=True)
radar_keys   = ["cardio_score", "sleep_duration_score", "fruitveg_score",
                "weights_score", "community_time_score", "checkup_score"]
radar_labels = ["Cardio", "Sleep", "Fruit/Veg", "Weights", "Community", "Checkups"]
N = len(radar_keys)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for name, color in zip(pnames, pcolors):
    vals_r  = [all_feats[name].get(k, 0) for k in radar_keys]
    vals_r += vals_r[:1]
    ax6.plot(angles, vals_r, color=color, lw=2)
    ax6.fill(angles, vals_r, color=color, alpha=0.1)

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(radar_labels, size=8)
ax6.set_ylim(0, 3)
ax6.set_yticks([1, 2, 3])
ax6.set_yticklabels(["1", "2", "3"], size=7, color="#888")
ax6.set_title("Good-Factor Radar", pad=15)
ax6.grid(color="#333")

plt.savefig(OUT / "plot_profile_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Plot 6 saved")
print(f"\n✓ Section 12 complete — all 6 plots saved to {OUT}/")